In [ ]:
# ============================================================
# 🦾 MASTER DASHBOARD: 2-Link Arm (Report + Replay)
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import glob
import os
import re

# 1. Setup
BASE_LOG_DIR = os.path.join("..", "..", "logs")
L1 = 15.0 # cm
L2 = 15.0 # cm
MAX_ALLOWED_ERROR = 0.5 # cm

def load_and_analyze(run_folder_path):
    clear_output(wait=True)
    run_name = os.path.basename(run_folder_path)
    
    # --- LOAD DATA ---
    log_files = glob.glob(os.path.join(run_folder_path, "arm_2link_log.csv"))
    if not log_files:
        print(f"❌ No data found in {run_name}")
        return
    df = pd.read_csv(log_files[0])
    df['time_sec'] = df['timestamp_ms'] / 1000.0
    
    # --- PRE-CALCULATE GEOMETRY (For Smooth Slider) ---
    rad_m1 = np.radians(df['motor1_pos'])
    df['elbow_x'] = L1 * np.cos(rad_m1)
    df['elbow_y'] = L1 * np.sin(rad_m1)
    
    # --- METRICS CALCULATION ---
    duration = df['time_sec'].max()
    max_error = df['error'].max()
    avg_error = df['error'].mean()
    final_error = df['error'].iloc[-1]
    verdict = "PASS" if max_error <= MAX_ALLOWED_ERROR else "FAIL"
    
    # 1. Total Travel (Odometer)
    total_j1_travel = np.sum(np.abs(np.diff(df['motor1_pos'])))
    total_j2_travel = np.sum(np.abs(np.diff(df['motor2_pos'])))

    # 2. Max Velocity Calculation (deg/s)
    # We calculate the difference in position divided by difference in time
    dt = df['time_sec'].diff().fillna(0.001) # Avoid division by zero
    j1_vel = np.abs(df['motor1_pos'].diff()) / dt
    j2_vel = np.abs(df['motor2_pos'].diff()) / dt
    
    # Filter out initial startup spikes (first 5 frames) to be safe
    max_j1_vel = j1_vel.iloc[5:].max() if len(j1_vel) > 5 else 0
    max_j2_vel = j2_vel.iloc[5:].max() if len(j2_vel) > 5 else 0

    # ========================================================
    # 📊 PART 1: STATIC ANALYSIS REPORT
    # ========================================================
    fig = plt.figure(figsize=(12, 8)) # Adjusted height since we removed a plot
    plt.subplots_adjust(hspace=0.3)
    
    # Plot A: The Map
    ax1 = fig.add_subplot(2, 1, 1) # Changed to 2 rows
    ax1.set_title(f"1. Cartesian Path ({run_name})", fontweight='bold')
    ax1.plot(df['target_x'], df['target_y'], 'g--', label='Target')
    ax1.plot(df['real_x'], df['real_y'], 'b-', label='Actual Tip')
    last = df.iloc[-1]
    ax1.plot([0, last['elbow_x'], last['real_x']], 
             [0, last['elbow_y'], last['real_y']], 'r-o', lw=2, label='Final Pose')
    ax1.scatter([0], [0], c='k', s=100, marker='s')
    ax1.axis('equal')
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper right')
    ax1.set_ylabel("Y (cm)")

    # Plot B: Error Timeline
    ax2 = fig.add_subplot(2, 1, 2) # Changed to 2 rows
    ax2.set_title("2. Tracking Error (Full Run)", fontweight='bold')
    ax2.plot(df['time_sec'], df['error'], 'r-', label='Error')
    ax2.axhline(MAX_ALLOWED_ERROR, color='k', linestyle=':', label='Limit')
    ax2.fill_between(df['time_sec'], df['error'], color='red', alpha=0.1)
    ax2.set_ylabel("Error (cm)")
    ax2.grid(True, alpha=0.3)
    
    plt.show()

    # --- SUMMARY TABLE ---
    summary_data = {
        "Metric": [
            "Duration", "Max Error", "Avg Error", "Verdict", 
            "---",
            "J1 Total Travel", "J2 Total Travel",
            "J1 Max Speed", "J2 Max Speed"
        ],
        "Value": [
            f"{duration:.2f} s", f"{max_error:.4f} cm", f"{avg_error:.4f} cm", verdict,
            "---",
            f"{total_j1_travel:.1f}°", f"{total_j2_travel:.1f}°",
            f"{max_j1_vel:.1f} deg/s", f"{max_j2_vel:.1f} deg/s"
        ]
    }
    print("\n" + "="*60)
    print(f"📊 REPORT: {run_name}")
    print("="*60)
    print(pd.DataFrame(summary_data).to_string(index=False))
    print("="*60 + "\n")

    # ========================================================
    # 🎮 PART 2: INTERACTIVE REPLAY
    # ========================================================
    print("👇 INTERACTIVE REPLAY (Drag Slider to See Motion History) 👇")
    
    def plot_replay_frame(frame_index):
        row = df.iloc[frame_index]
        history_slice = df.iloc[:frame_index+1:5]
        
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.set_xlim(-10, 35)
        ax.set_ylim(-15, 25)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        
        ax.set_title(f"Time: {row['time_sec']:.2f}s | J1: {row['motor1_pos']:.1f}° | J2: {row['motor2_pos']:.1f}°")
        
        # 1. Paths
        ax.plot(df['target_x'], df['target_y'], 'g--', alpha=0.2, label="Mission Path")
        ax.plot(history_slice['real_x'], history_slice['real_y'], 'b-', linewidth=2, alpha=0.6, label="Tip Trace")
        ax.plot(history_slice['elbow_x'], history_slice['elbow_y'], color='orange', linestyle=':', linewidth=2, alpha=0.5, label="Elbow Trace")
        
        # 2. Arm
        ax.plot([0, row['elbow_x'], row['real_x']], 
                [0, row['elbow_y'], row['real_y']], 
                'r-o', lw=3, solid_capstyle='round', label="Arm")
        
        # 3. Base & Labels
        ax.scatter([0], [0], c='k', s=150, marker='s')
        ax.text(0, -2, "J1 (Base)", ha='center', fontweight='bold')
        ax.text(row['elbow_x'], row['elbow_y'] + 2, "J2", ha='center', fontweight='bold', color='darkred')
        
        ax.legend(loc="upper right")
        plt.show()

    slider = widgets.IntSlider(
        min=0, max=len(df)-1, step=10, value=0,
        description='Scrub:', layout=widgets.Layout(width='100%')
    )
    widgets.interact(plot_replay_frame, frame_index=slider)

# --- FOLDER DROPDOWN ---
all_folders = [f.path for f in os.scandir(BASE_LOG_DIR) if f.is_dir()]
arm_folders = sorted(
    [f for f in all_folders if "arm_run" in os.path.basename(f)],
    key=lambda x: int(re.search(r"arm_run_?(\d+)", os.path.basename(x)).group(1)) 
                  if re.search(r"arm_run_?(\d+)", os.path.basename(x)) else 0
)

if not arm_folders:
    print("❌ No 'arm_run' folders found.")
else:
    w = widgets.Dropdown(options=[(os.path.basename(f), f) for f in arm_folders], description='Select Run:')
    widgets.interact(load_and_analyze, run_folder_path=w)

interactive(children=(Dropdown(description='Select Run:', options=(('arm_run_1', '..\\..\\logs\\arm_run_1'), (…